Думаю будет очевидно, что я взял лучшую модель LightGBM. Алгоритм будет таким же, но без удаления данных: просто приведение данных к необходимым столбцам, потому что модель тренировалась именно на них.

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
test = pd.read_csv("test_real_estate_samples.csv")
b_sale = joblib.load("../01.09.26 - 14.09.26/model_sale.joblib")
b_rent = joblib.load("../01.09.26 - 14.09.26/model_rental.joblib")

In [3]:
test["size"] = test["size"].replace(1.0, np.nan).fillna(100.0)
test["room_count"] = test["room_count"].replace("+", "2+1")

In [4]:
df = test.copy()
addr = df["address"].str.split("/", expand=True)
df["city"] = addr[0]
df["district"] = addr[1]

In [5]:
rooms = df["room_count"].str.split("+", expand=True).astype(int)
df["total_rooms"] = rooms[0] + rooms[1]
df["living_rooms_count"] = rooms[1]
df["area_per_room"] = (df["size"] / df["total_rooms"]).round(2)

In [6]:
age_map = {
    "6-10 arası": 8,
    "11-15 arası": 13,
    "16-20 arası": 18,
    "21-25 arası": 23,
    "26-30 arası": 28,
    "31-35 arası": 33,
    "36-40 arası": 38,
    "40 ve üzeri": 42,
}
floor_map = {
    "10-20 arası": 15,
    "20 ve üzeri": 20,
    "Yüksek Giriş": 1,
    "Asma Kat": 1,
    "Müstakil": 1,
    "Komple": 1,
    "Zemin Kat": 0,
    "Giriş Katı": 0,
    "Bahçe katı": 0,
    "Bodrum Kat": -1,
    "Kot 1": -1,
    "Kot 2": -2,
    "Kot 3": -3,
    "Kot 4": -4,
    "En Üst Kat": 4,
    "Çatı Katı": 4,
    "Teras Kat": 4,
}

df["is_age_unknown"] = (
    df["building_age"].isna() | (df["building_age"] == "Unknown")
).astype(int)
df["building_age"] = (
    pd.to_numeric(df["building_age"].replace(age_map), errors="coerce")
    .fillna(10)
    .astype(int)
)

In [7]:
df["total_floor_count"] = (
    pd.to_numeric(df["total_floor_count"].replace(floor_map), errors="coerce")
    .fillna(4)
    .astype(int)
)
df["floor_no"] = (
    pd.to_numeric(df["floor_no"].replace(floor_map), errors="coerce")
    .fillna(1)
    .astype(int)
)
df["total_floor_count"] = np.maximum(df["total_floor_count"], df["floor_no"])

In [8]:
df["floor_ratio"] = (
    (df["floor_no"] / df["total_floor_count"]).clip(0, 1).round(2)
)
df["is_first_floor"] = (df["floor_no"] <= 0).astype(int)
df["is_top_floor"] = (df["floor_no"] == df["total_floor_count"]).astype(int)

In [9]:
preds = pd.Series(0.0, index=df.index)

for l_type, bundle in [(1, b_sale), (2, b_rent)]:
    mask = df["listing_type"] == l_type
    sub = df[mask].copy()

    sub["district"] = bundle["target_encoder"].transform(sub[["district"]])
    sub = pd.get_dummies(sub, columns=["sub_type", "heating_type", "city"])

    X = sub.reindex(columns=bundle["feature_names"], fill_value=0)
    preds[mask] = bundle["model"].predict(X)

In [10]:
res = test.copy()
res.insert(0, "ФИО", "Разумченко Д. В.")
res.insert(1, "Группа", "23П-1")
res["price_pred"] = preds.round(2)
res.to_csv("result_real_estate_samples.csv", index=False, encoding="utf-8-sig")

In [11]:
y, p = res["price"].values, preds.values

In [12]:
mae_m = np.mean(np.abs(y - p))
rmse_m = np.sqrt(np.mean((y - p) ** 2))
r2_m = 1 - np.sum((y - p) ** 2) / np.sum((y - np.mean(y)) ** 2)

In [13]:
metrics_all = pd.DataFrame(
    {
        "Метрика": ["MAE", "RMSE", "R²"],
        "Вручную": [f"{mae_m:,.2f}", f"{rmse_m:,.2f}", f"{r2_m:.4f}"],
        "Sklearn": [
            f"{mean_absolute_error(y, p):,.2f}",
            f"{np.sqrt(mean_squared_error(y, p)):,.2f}",
            f"{r2_score(y, p):.4f}",
        ],
    }
)

In [14]:
mask_s, mask_r = res["listing_type"] == 1, res["listing_type"] == 2

In [15]:
def get_seg_metrics(y_true, y_pred):
    return (
        f"{mean_absolute_error(y_true, y_pred):,.2f}",
        f"{np.sqrt(mean_squared_error(y_true, y_pred)):,.2f}",
        f"{r2_score(y_true, y_pred):.4f}",
    )

In [16]:
s_mae, s_rmse, s_r2 = get_seg_metrics(y[mask_s], p[mask_s])
r_mae, r_rmse, r_r2 = get_seg_metrics(y[mask_r], p[mask_r])

In [17]:
metrics_seg = pd.DataFrame(
    {
        "Сегмент": ["Продажа (listing_type=1)", "Аренда (listing_type=2)"],
        "Кол-во": [mask_s.sum(), mask_r.sum()],
        "MAE (TRY)": [s_mae, r_mae],
        "RMSE (TRY)": [s_rmse, r_rmse],
        "R²": [s_r2, r_r2],
    }
)

In [18]:
print("--- СРАВНЕНИЕ РАСЧЕТА МЕТРИК ---")
print(metrics_all.to_string(index=False))
print("\n--- МЕТРИКИ ПО СЕГМЕНТАМ ---")
print(metrics_seg.to_string(index=False))
print("\n--- СОПОСТАВЛЕНИЕ ФАКТА И ПРОГНОЗА ---")
cols_show = [
    "id",
    "listing_type",
    "sub_type",
    "size",
    "room_count",
    "price",
    "price_pred",
]
print(res[cols_show].to_string(index=False))

--- СРАВНЕНИЕ РАСЧЕТА МЕТРИК ---
Метрика    Вручную    Sklearn
    MAE 160,881.75 160,881.75
   RMSE 353,443.22 353,443.22
     R²     0.6793     0.6793

--- МЕТРИКИ ПО СЕГМЕНТАМ ---
                 Сегмент  Кол-во  MAE (TRY) RMSE (TRY)      R²
Продажа (listing_type=1)      10 316,746.90 499,780.87  0.4996
 Аренда (listing_type=2)      10   5,016.60   7,955.99 -0.6598

--- СОПОСТАВЛЕНИЕ ФАКТА И ПРОГНОЗА ---
   id  listing_type     sub_type  size room_count     price  price_pred
   49             1   Kooperatif 100.0        1+0  300000.0   360839.28
  506             1  Çiftlik Evi 250.0        2+2 1600000.0   762272.02
    2             1        Daire  43.0        1+0  490000.0   364958.55
  528             1 Prefabrik Ev 122.0        3+1  103600.0   179126.63
  109             1   Kooperatif 100.0        1+0  300000.0   360839.28
  100             1       Yazlık 150.0        3+1  500000.0   332862.89
  320             1       Yazlık 134.0        4+1 1000000.0   598710.90
    3       

В тестовом датасете присутствуют редкие типы недвижимости и пропуски в ключевых факторах цены, что сильно сказывается на итоговом предсказании (а так же на метрики). Однако, это проблема данных, а не модели.

Стоит заметить, что на хороших данных предсказание было практически идеальным (49, 2, 109, 3, 7810, 1, 7, 58, 16).